# Week 3 · Class 2 — Exploring wearable data with Plotly 📈⌚

**Goal of today (60 min):** answer real questions about a dataset with the *right* chart, and make those charts look professional.

We work with a dataset like the ones health platforms collect from wearables:
**250 users × 365 days = 91,250 rows** of daily steps, sleep, heart rate, HRV and stress, from 8 European cities and 4 devices.

| Part | Question we ask | Charts |
|---|---|---|
| 0 | What does the data look like? Is it clean? | missing-values bar, box |
| 1 | How are values distributed? | histogram, violin, ECDF |
| 2 | How do groups compare? | bar (sorted, grouped, highlighted) |
| 3 | Are variables related? | scatter + trendline, scatter matrix, heatmap |
| 4 | How do things change over time? | line, rolling mean, range slider, calendar heatmap |
| 5 | What is the composition? | sunburst, treemap |
| 6 | Where? | bubble map |
| 7 | How does it evolve? | animation |
| 8 | Make it presentable | templates, annotations, subplots |
| 9 | From notebook to app | wrap a chart in a function |

> 💡 Rule of the day: **first the question, then the chart.**

## ⚙️ Setup

📁 Put `wearables_daily.csv` next to this notebook (in **Colab**: drag it into the Files panel on the left), then run the two cells below.

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# In Colab charts render automatically. If they don't show in VS Code/Jupyter, uncomment:
# import plotly.io as pio; pio.renderers.default = "notebook"

In [3]:
df = pd.read_csv("../data/wearables_daily.csv", parse_dates=["date"])   # parse_dates: read 'date' as a real date
df.head()

,user_id,date,city,device,age,gender,steps,active_minutes,sleep_hours,deep_sleep_pct,resting_hr,hrv_ms,stress_score,calories,age_group,lat,lon
0,U001,2025-10-01,Berlin,Apple Watch,58,female,11208,91.0,6.24,12.8,68.0,38.7,56.0,2412.0,50-59,52.52,13.4
1,U001,2025-10-02,Berlin,Apple Watch,58,female,14360,123.0,5.86,17.7,65.0,38.6,56.0,2666.0,50-59,52.52,13.4
2,U001,2025-10-03,Berlin,Apple Watch,58,female,10323,81.0,6.95,15.6,65.0,31.5,56.0,2337.0,50-59,52.52,13.4
3,U001,2025-10-04,Berlin,Apple Watch,58,female,10048,87.0,7.06,24.8,65.0,30.5,41.0,2350.0,50-59,52.52,13.4
4,U001,2025-10-05,Berlin,Apple Watch,58,female,10675,98.0,6.76,12.8,64.0,39.0,59.0,2419.0,50-59,52.52,13.4


## 0. First look: what do we have? Is it clean? 🔍

Before plotting anything, we always check **size, types, missing values and weird values**.

In [4]:
# ✍️ live coding
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 91250 entries, 0 to 91249
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   user_id         91250 non-null  str           
 1   date            91250 non-null  datetime64[us]
 2   city            91250 non-null  str           
 3   device          91250 non-null  str           
 4   age             91250 non-null  int64         
 5   gender          91250 non-null  str           
 6   steps           91250 non-null  int64         
 7   active_minutes  91250 non-null  float64       
 8   sleep_hours     88514 non-null  float64       
 9   deep_sleep_pct  88514 non-null  float64       
 10  resting_hr      91250 non-null  float64       
 11  hrv_ms          83017 non-null  float64       
 12  stress_score    91250 non-null  float64       
 13  calories        91250 non-null  float64       
 14  age_group       91250 non-null  str           
 15  lat          

In [5]:
# ✍️ live coding
df.describe().round(1)


,date,age,steps,active_minutes,sleep_hours,deep_sleep_pct,resting_hr,hrv_ms,stress_score,calories,lat,lon
count,91250,91250.0,91250.0,91250.0,88514.0,88514.0,91250.0,83017.0,91250.0,91250.0,91250.0,91250.0
mean,2026-04-01 00:00:00,44.4,8376.5,69.4,7.2,17.6,71.7,44.3,50.5,2210.9,47.8,5.4
min,2025-10-01 00:00:00,20.0,1179.0,0.0,4.2,5.0,44.0,10.0,6.0,1659.0,38.7,-9.1
25%,2025-12-31 00:00:00,32.0,5671.0,47.0,6.7,14.7,68.0,34.7,43.0,2015.0,40.4,-3.7
50%,2026-04-01 00:00:00,44.0,7572.0,64.0,7.2,17.6,72.0,44.2,51.0,2156.0,48.9,4.9
75%,2026-07-01 00:00:00,57.0,10121.0,85.0,7.8,20.4,75.0,53.9,58.0,2344.0,52.4,13.4
max,2026-09-30 00:00:00,69.0,94771.0,387.0,10.9,35.0,90.0,92.2,100.0,5017.0,55.7,16.4
std,NaN,14.4,4213.4,32.8,0.8,4.3,5.2,13.1,11.4,281.2,5.5,8.5


👀 Look at `steps`: the median is ~7,500 but the **max is above 90,000**. Nobody walks 90k steps a day — that smells like a sensor glitch.

Let's **see** the missing values first. A bar chart of `isna().sum()` is the quickest data-quality chart.

In [6]:
# ✍️ live coding
import sys; print(sys.executable)



/Users/andredeoliveira/wearables_dashboard/.venv/bin/python


Is the missing HRV random, or does it depend on the device? 🤔

In [7]:
# ✍️ live coding
## columns missing values
missing = df.isna().mean().mul(100).round(1).sort_values()
missing = missing[missing > 0].reset_index()
missing.columns = ["column", "pct_missing"]

fig = px.bar(missing, x="pct_missing", y="column", orientation="h", text="pct_missing",
             title="Which columns have missing values? (%)")
fig.show()


💡 **Insight:** missing HRV is *not random* — it comes from one device. Good to know before averaging HRV per device!

Now the outliers. A **box plot** shows them instantly as isolated points.

In [ ]:
# ✍️ live coding


In [9]:
# ✍️ live coding
px.box(df, x= "steps", title = "Steps per day")


## 1. Distributions: what values are typical? 📊

| Chart | When to use |
|---|---|
| Histogram | shape of one numeric variable |
| Histogram + `marginal` | shape **and** outliers in one figure |
| Overlaid histograms / violins | compare the shape between groups |
| ECDF | "what % of days are below X?" |

In [11]:
# ✍️ live coding

px.histogram(df, x ="sleep_hours", nbins =60, marginal="box",
             title = "How long do people sleep")

In [12]:
# ✍️ live coding
px.histogram(df, x="steps", color="device", barmode="overlay", opacity=0.6,
             histnorm="probability density", nbins=80,
             title="Steps distribution per device")


`histnorm='probability density'` lets us compare groups of **different sizes** fairly.

In [13]:
# ✍️ live coding

age_order = ["20-29", "30-39", "40-49", "50-59", "60+"]
px.violin(df, x="age_group", y="resting_hr", box=True, color="age_group",
          category_orders={"age_group": age_order},
          title="Resting heart rate by age group")

In [ ]:
# ✍️ live coding


💡 Read the ECDF at 10,000 steps: what % of days do **not** reach the famous 10k goal?

## 2. Comparing groups 🏆

Bar charts are the workhorse. Three tricks make them much better:
1. **Sort** the bars.
2. Go **horizontal** when labels are long or many.
3. **Highlight** what matters, grey out the rest.

In [18]:
# ✍️ live coding
by_city = df.groupby("city", as_index=False)["steps"].mean()
by_city
px.bar(by_city, x="steps", y="city", orientation="h", text_auto=".0f",
       title="Average daily steps per city")

In [19]:
per_user = df.groupby(["user_id", "age_group", "device", "gender"], as_index=False)[
    ["steps", "sleep_hours", "resting_hr", "hrv_ms", "stress_score", "age"]].mean()
per_user.head()

,user_id,age_group,device,gender,steps,sleep_hours,resting_hr,hrv_ms,stress_score,age
0,U001,50-59,Apple Watch,female,10765.358904,6.378719,66.339726,32.345479,54.709589,58.0
1,U002,60+,Fitbit,male,14368.309589,8.094628,62.745205,31.736986,41.989041,65.0
2,U003,40-49,Withings,female,6835.189041,7.930084,74.498630,47.422477,46.361644,43.0
3,U004,60+,Garmin,female,6262.079452,6.465057,77.046575,29.873151,56.265753,62.0
4,U005,50-59,Garmin,female,3487.712329,7.101289,78.128767,38.926849,53.572603,51.0


In [20]:
# ✍️ live coding
px.scatter(per_user, x="steps", y="resting_hr", trendline="ols",
           hover_name="user_id", hover_data=["age_group", "device"],
           title="More steps → lower resting heart rate?")


😴 Weekends are visible. But the y-axis starts at 0, so the differences look tiny — is that the right choice here? Bars **must** start at 0. If you want to zoom in on differences, use dots or lines instead.

In [21]:
# ✍️ live coding
px.scatter_matrix(per_user, dimensions=["steps", "sleep_hours", "resting_hr", "hrv_ms", "stress_score"],
                  color="age_group", opacity=0.6, height=750,
                  title="All pairs at once")


## 3. Relationships between variables 🔗

With 90k rows a raw scatter becomes a blob. **Aggregate first** (one point per user) — a very common EDA move.

In [22]:
# ✍️ live coding
px.treemap(per_user, path=["device", "age_group"], values="steps", color="sleep_hours",
           color_continuous_scale="Viridis",
           title="Total steps by device and age (colour = sleep)")

In [ ]:
# ✍️ live coding


`trendline='ols'` fits a linear regression (needs `statsmodels`). Hover on the line to see the equation and R².

In [23]:
# ✍️ live coding
per_city = df.groupby(["city", "lat", "lon"], as_index=False).agg(
    users=("user_id", "nunique"), steps=("steps", "mean"), sleep=("sleep_hours", "mean"))

px.scatter_map(per_city, lat="lat", lon="lon", size="users", color="steps",
               hover_name="city", hover_data={"sleep": ":.2f", "lat": False, "lon": False},
               size_max=40, zoom=3, map_style="carto-positron",
               color_continuous_scale="Tealgrn", title="Users and activity per city")


In [25]:
df

,user_id,date,city,device,age,gender,steps,active_minutes,sleep_hours,deep_sleep_pct,resting_hr,hrv_ms,stress_score,calories,age_group,lat,lon
0,U001,2025-10-01,Berlin,Apple Watch,58,female,11208,91.0,6.24,12.8,68.0,38.7,56.0,2412.0,50-59,52.52,13.4
1,U001,2025-10-02,Berlin,Apple Watch,58,female,14360,123.0,5.86,17.7,65.0,38.6,56.0,2666.0,50-59,52.52,13.4
2,U001,2025-10-03,Berlin,Apple Watch,58,female,10323,81.0,6.95,15.6,65.0,31.5,56.0,2337.0,50-59,52.52,13.4
3,U001,2025-10-04,Berlin,Apple Watch,58,female,10048,87.0,7.06,24.8,65.0,30.5,41.0,2350.0,50-59,52.52,13.4
4,U001,2025-10-05,Berlin,Apple Watch,58,female,10675,98.0,6.76,12.8,64.0,39.0,59.0,2419.0,50-59,52.52,13.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91245,U250,2026-09-26,Amsterdam,Withings,59,male,10582,85.0,7.07,15.1,67.0,31.5,57.0,2363.0,50-59,52.37,4.9
91246,U250,2026-09-27,Amsterdam,Withings,59,male,7547,72.0,8.01,10.1,65.0,NaN,39.0,2190.0,50-59,52.37,4.9
91247,U250,2026-09-28,Amsterdam,Withings,59,male,21252,184.0,7.22,15.9,70.0,34.1,49.0,3186.0,50-59,52.37,4.9
91248,U250,2026-09-29,Amsterdam,Withings,59,male,20826,173.0,7.52,27.3,63.0,NaN,35.0,3125.0,50-59,52.37,4.9


In [28]:
df["month"] = df["date"].dt.month
df["year_month"] = df["date"].dt.to_period("M").astype(str)
df

,user_id,date,city,device,age,gender,steps,active_minutes,sleep_hours,deep_sleep_pct,resting_hr,hrv_ms,stress_score,calories,age_group,lat,lon,month,year_month
0,U001,2025-10-01,Berlin,Apple Watch,58,female,11208,91.0,6.24,12.8,68.0,38.7,56.0,2412.0,50-59,52.52,13.4,10,2025-10
1,U001,2025-10-02,Berlin,Apple Watch,58,female,14360,123.0,5.86,17.7,65.0,38.6,56.0,2666.0,50-59,52.52,13.4,10,2025-10
2,U001,2025-10-03,Berlin,Apple Watch,58,female,10323,81.0,6.95,15.6,65.0,31.5,56.0,2337.0,50-59,52.52,13.4,10,2025-10
3,U001,2025-10-04,Berlin,Apple Watch,58,female,10048,87.0,7.06,24.8,65.0,30.5,41.0,2350.0,50-59,52.52,13.4,10,2025-10
4,U001,2025-10-05,Berlin,Apple Watch,58,female,10675,98.0,6.76,12.8,64.0,39.0,59.0,2419.0,50-59,52.52,13.4,10,2025-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91245,U250,2026-09-26,Amsterdam,Withings,59,male,10582,85.0,7.07,15.1,67.0,31.5,57.0,2363.0,50-59,52.37,4.9,9,2026-09
91246,U250,2026-09-27,Amsterdam,Withings,59,male,7547,72.0,8.01,10.1,65.0,NaN,39.0,2190.0,50-59,52.37,4.9,9,2026-09
91247,U250,2026-09-28,Amsterdam,Withings,59,male,21252,184.0,7.22,15.9,70.0,34.1,49.0,3186.0,50-59,52.37,4.9,9,2026-09
91248,U250,2026-09-29,Amsterdam,Withings,59,male,20826,173.0,7.52,27.3,63.0,NaN,35.0,3125.0,50-59,52.37,4.9,9,2026-09


In [30]:
# ✍️ live coding
city_month = df.groupby(["year_month", "city"], as_index=False).agg(
    steps=("steps", "mean"), sleep=("sleep_hours", "mean"), users=("user_id", "nunique"))

px.scatter(city_month, x="steps", y="sleep", size="users", color="city", text="city",
           animation_frame="year_month", range_x=[6500, 10500], range_y=[6.9, 7.65],
           size_max=45, title="Cities through the year: steps vs sleep")


In [ ]:
# ✍️ live coding


💡 Correlations go from −1 to +1, so we use a **diverging** colour scale centred at 0 (remember the colour-scale lesson!).

## 4. Time series ⏳

Daily data is noisy. A **rolling mean** shows the trend; a **range slider** lets the reader zoom.

In [ ]:
# ✍️ live coding


☀️ A clear **summer peak** and a **weekly zig-zag** (weekends). Let's look at the weekly pattern with a calendar heatmap.

In [ ]:
# ✍️ live coding


In [ ]:
# ✍️ live coding


## 5. Part-to-whole 🥧➡️🌞

Pie charts fail with many slices. For **hierarchies** use a sunburst or a treemap.

In [ ]:
# ✍️ live coding


In [ ]:
# ✍️ live coding


## 6. Geographical data 🗺️

`px.scatter_map` draws on an open map — **no token needed** (it replaces the old `scatter_mapbox`).

In [ ]:
# ✍️ live coding


## 7. Animations 🎬

An animation adds the **time** dimension to a scatter. Press ▶️.

In [ ]:
# ✍️ live coding


⚠️ Always fix `range_x` / `range_y` in animations, otherwise the axes jump at every frame.

## 8. From exploratory to presentation-ready ✨

Exploratory charts are fast and ugly; presentation charts must be **understood in 3 seconds**.
Tools: **templates**, clear **titles and labels**, **annotations**, **reference lines**.

In [ ]:
# ✍️ live coding


In [ ]:
# ✍️ live coding


💡 The **title tells the insight**, the subtitle tells the data. Now let's combine several charts with `make_subplots` — a mini dashboard!

In [ ]:
# ✍️ live coding


It works… but it is static. To let users **choose** the city, device or metric, we need an **app**. That is Dash — next part!

## 9. From notebook to app: wrap charts in functions 🧩

In the project, every chart will be a **function**: data goes in, figure comes out.
This makes it reusable — the app will call it every time a user changes a filter.

In [ ]:
# ✍️ live coding


In [ ]:
# ✍️ live coding


## 🎯 Recap

- **Question first**, chart second.
- EDA starts with **quality checks**: missing values (are they random?) and outliers.
- Distributions → histogram/violin/ECDF · Comparisons → sorted bars · Relationships → scatter + trendline, heatmap · Time → rolling mean · Hierarchy → sunburst/treemap · Space → map.
- **Aggregate** big data before plotting.
- Presentation charts: insight as title, labels, annotations, consistent template.
- Charts in **functions** = ready for a dashboard. 🚀

### 📚 To go further
- [Plotly Express reference](https://plotly.com/python-api-reference/plotly.express.html)
- [Built-in colour scales](https://plotly.com/python/builtin-colorscales/)
- [Templates & theming](https://plotly.com/python/templates/)
- [Dash tutorial](https://dash.plotly.com/tutorial)